<a href="https://colab.research.google.com/github/CorgiFan/Pytorch_Practice/blob/main/DNA_Methylation_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("juanschafle/dna-methylation-data-epigenetic-biomarkers")

print("Path to dataset files:", path)

100%|██████████| 69.7k/69.7k [00:00<00:00, 357kB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/juanschafle/dna-methylation-data-epigenetic-biomarkers/versions/1


In [ ]:
import os

print(os.listdir(path))

['train_data.csv', 'test_data.csv', 'full_dataset.csv']


In [ ]:
import pandas as pd
train_data = path + '/train_data.csv'
test_data = path + '/test_data.csv'
train_df = pd.read_csv(train_data)
test_df = pd.read_csv(test_data)

train_X = train_df.drop('methylation_status', axis = 1)
train_y = train_df['methylation_status']
test_X = test_df.drop('methylation_status', axis = 1)
test_y = test_df['methylation_status']
print(train_X.shape)

(800, 4)


In [ ]:
class DNAModel(nn.Module):
    def __init__(self):
      super().__init__()
      self.layers = nn.Sequential(
          nn.Linear(4, 32),
          nn.ReLU(),
          nn.Linear(32, 16),
          nn.ReLU(),
          nn.Linear(16, 1)
      )
    def forward(self, x):
      x = self.layers(x)
      return x

In [ ]:
X_train = torch.tensor(train_X.values, dtype=torch.float32)
y_train = torch.tensor(train_y.values, dtype=torch.float32).view(-1, 1)

X_test = torch.tensor(test_X.values, dtype=torch.float32)
y_test = torch.tensor(test_y.values, dtype=torch.float32).view(-1, 1)

In [ ]:
model = DNAModel()

criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
best_accuracy = 0.0

for epoch in range(epochs):

    model.train()

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            outputs = model(X_batch)

            predictions = (torch.sigmoid(outputs) >= 0.5).float()

            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)

    accuracy = correct / total

    print(f"Epoch {epoch+1}: Accuracy = {accuracy:.4f}")

    # Save the best model
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save(model.state_dict(), "best_model.pth")

Epoch 1: Accuracy = 0.9850
Epoch 2: Accuracy = 0.9800
Epoch 3: Accuracy = 0.9650
Epoch 4: Accuracy = 0.9750
Epoch 5: Accuracy = 0.9800
Epoch 6: Accuracy = 0.9850
Epoch 7: Accuracy = 0.9850
Epoch 8: Accuracy = 0.9850
Epoch 9: Accuracy = 0.9750
Epoch 10: Accuracy = 0.9750
Epoch 11: Accuracy = 0.9750
Epoch 12: Accuracy = 0.9750
Epoch 13: Accuracy = 0.9850
Epoch 14: Accuracy = 0.9750
Epoch 15: Accuracy = 0.9900
Epoch 16: Accuracy = 0.9850
Epoch 17: Accuracy = 0.9850
Epoch 18: Accuracy = 0.9850
Epoch 19: Accuracy = 0.9700
Epoch 20: Accuracy = 0.9750
Epoch 21: Accuracy = 0.9850
Epoch 22: Accuracy = 0.9850
Epoch 23: Accuracy = 0.9800
Epoch 24: Accuracy = 0.9800
Epoch 25: Accuracy = 0.9900
Epoch 26: Accuracy = 0.9800
Epoch 27: Accuracy = 0.9800
Epoch 28: Accuracy = 0.9900
Epoch 29: Accuracy = 0.9700
Epoch 30: Accuracy = 0.9850
Epoch 31: Accuracy = 0.9750
Epoch 32: Accuracy = 0.9850
Epoch 33: Accuracy = 0.9750
Epoch 34: Accuracy = 0.9800
Epoch 35: Accuracy = 0.9800
Epoch 36: Accuracy = 0.9850
E

In [ ]:
torch.save(model.state_dict(), "final_model.pth")